# LFC Integration Workflow

Use this notebook when biological replicates are **not paired** across data types  
(e.g. TX and MX samples were collected independently, or from different organisms).

**Pipeline:**
```
linked_data
  → filter_data          (remove rare features)
  → devariance_data      (remove low-variance features)
  → replicate_filtering  (remove outlier replicates)
  → integrate_data       (LFC of group medians → features × contrasts)
  → feature_selection    (lfc cutoff or variance; sample-level methods not valid here)
  → feature_grouping     (group features by LFC pattern similarity)
  → compare_groups_to_pathways
```

**Config files used:**
- `config_examples/data_processing.yml` — filtering, devariancing, replicate handling
- `config_examples/analysis_lfc.yml` — LFC params, feature selection, feature grouping

# Project setup

In [ ]:
import os
import sys
import yaml
import pandas as pd
import tools.objects as objs
import tools.helpers as hlp
pd.options.display.max_columns = None
pd.set_option('display.max_colwidth', 500)

In [ ]:
# Optional: list all configurations available to load with the 'config_path' argument below
hlp.list_persistent_configs()

In [ ]:
# Initialize project (add the data processing and analysis hashes if resuming a previous project)
project = objs.Project(data_processing_hash=None, analysis_hash=None, overwrite=False)

# Dataset initiation

In [ ]:
# Create dataset objects and gather raw metadata and data (already pre-obtained from JGI sources)
tx_dataset = objs.TX(project)
mx_dataset = objs.MX(project, last=True)

# Analysis initiation

In [ ]:
# Create analysis object (collection of datasets and methods for performing integration)
# integration_mode is automatically detected as 'lfc' from analysis_lfc.yml
analysis = objs.Analysis(project, datasets=[tx_dataset, mx_dataset])
print(f"integration_mode: {analysis.integration_mode}")

# Link Samples Between Datasets

In [ ]:
# Link analysis datasets by finding corresponding sample metadata fields
analysis.link_metadata()

In [ ]:
# Link analysis datasets matrices by using linked metadata
analysis.link_data()

# Data processing

Steps shared by both branches. Parameters are set in `data_processing.yml`.

In [ ]:
# Filter out rare features from analysis datasets based on minimum observed value
analysis.filter_all_datasets()

In [ ]:
# Filter out features from analysis datasets that were not impacted by experimentation
analysis.devariance_all_datasets()

In [ ]:
# Filter out features with outlier replicates from each dataset
analysis.replicability_test_all_datasets()

In [ ]:
# View non-normalized data distributions
hlp.plot_simple_histogram(mx_dataset.replicate_filtered_data, plot_title="Metabolite Peak Heights", output_dir=analysis.output_dir)
hlp.plot_simple_histogram(tx_dataset.replicate_filtered_data, plot_title="Transcript Counts", output_dir=analysis.output_dir)

# Integration analysis

LFC of group medians is computed per dataset and the resulting contrast matrices are concatenated.  
Per-dataset scaling is **not** applied — LFC is scale-invariant.  
The output `integrated_data` is already in LFC space (features × contrasts).

In [ ]:
# Integrate metadata tables (produces contrast-level metadata: one row per group contrast)
analysis.integrate_metadata()

In [ ]:
# Integrate data matrices: compute LFC per dataset and concatenate
# Output: integrated_data_selected (features × contrasts) — ready for downstream analysis
analysis.integrate_data()

In [ ]:
# Annotate the integrated features with pre-generated feature annotation tables
analysis.annotate_integrated_features()

In [ ]:
# Subset features using LFC cutoff or variance (sample-level methods not valid in lfc mode)
analysis.perform_feature_selection()

In [ ]:
# View PCA of the LFC matrix to examine clustering pattern of combined data types
hlp.plot_simple_pca(
    analysis.integrated_data_selected,
    analysis.integrated_metadata,
    title="PCA of LFC integrated data",
    output_dir=analysis.output_dir
)

# Find data-driven groups of features

In [ ]:
# Group features by LFC pattern similarity across contrasts
analysis.group_features()

# Data-driven vs. knowledge-driven feature grouping

In [ ]:
# Compare data-driven feature groups against biological pathway annotations
grouping_results = analysis.compare_groups_to_pathways()

# Exploration of Results

In [ ]:
# Assess enrichment of an annotation layer in the extracted groups
#analysis.perform_functional_enrichment()

In [ ]:
# View average LFC abundance of a given group across contrasts
analysis.plot_submodule_avg_abundance(submodule_name='group_1', metadata_cat='group', save_plot=True)

In [ ]:
# View LFC patterns of individual features
analysis.plot_individual_feature(feature_id='mx_11781_positive', metadata_cat='group', save_plot=True)

In [ ]:
# Sync all results tables to database for query actions
analysis.register_all_existing_data()

# Save configuration and notebook

In [ ]:
# Save persistent configuration and notebook files
project.save_persistent_config_and_notebook()